## BI-RADS ViT pipeline (overview)

This notebook builds on **paired mammography views** (CC and MLO per breast) from `clean_pairs.csv`, with **patient-level** train/validation/test splits so the same patient does not appear in more than one split.

**Primary path — ordinal BI-RADS (2–5):** We treat the four BI-RADS categories as an **ordered** scale. The model is a ViT backbone with a **CORAL-style** head (`num_classes - 1` logits) and **ordinal loss** (thresholded cumulative binary targets). That preserves ordering between classes and is the main setup used for **full-category** grading.

We train **one ordinal model on CC** and **one on MLO** (same architecture, separate weights). At test time we also form an **ensemble** by averaging the two models’ **ordinal class probabilities** on paired batches (CC and MLO for the same row).

**Later in the notebook** we **change the objective**: separate **binary classifiers** (benign BI-RADS 2–3 vs malignant 4–5) with direct **binary cross-entropy** — including a **fused CC+MLO** model — so you can compare ordinal vs binary supervision without removing the ordinal experiments above.

In [34]:
import os
import glob
import re
import pandas as pd
from transformers import ViTModel

In [35]:
def generate_paired_manifest():
    base_dir = "birads_dataset"
    jpeg_dir = os.path.join(base_dir, "jpeg")
    output_csv = "birads_dataset/clean_pairs.csv"
    
    # 1. Map all available JPEGs
    print("Scanning for JPEG images...")
    all_jpegs = glob.glob(os.path.join(jpeg_dir, "**", "*.jpg"), recursive=True)
    uid_to_jpg = {os.path.basename(os.path.dirname(jpg)): jpg for jpg in all_jpegs}

    # 2. Load and Combine Metadata
    mass_csv = os.path.join(base_dir, "csv/mass_case_description_train_set.csv")
    calc_csv = os.path.join(base_dir, "csv/calc_case_description_train_set.csv")
    
    dfs = []
    if os.path.exists(mass_csv):
        # Fix column inconsistency
        m_df = pd.read_csv(mass_csv).rename(columns={'breast_density': 'breast density'})
        dfs.append(m_df)
    if os.path.exists(calc_csv):
        dfs.append(pd.read_csv(calc_csv))
        
    df = pd.concat(dfs, ignore_index=True)
    df.columns = df.columns.str.strip() 

    # 3. Match Metadata to JPEGs
    def get_jpg_path(dicom_path):
        if not isinstance(dicom_path, str): return None
        for uid, jpg_path in uid_to_jpg.items():
            if uid in dicom_path: return jpg_path
        return None

    df['jpg_path'] = df['image file path'].apply(get_jpg_path)
    df_clean = df.dropna(subset=['jpg_path'])

    # 4. FIX: Aggregate Abnormality Labels to Image Level FIRST
    # If an image has multiple abnormalities, take the max BI-RADS assessment
    img_level = df_clean.groupby(
        ['patient_id', 'left or right breast', 'image view', 'jpg_path']
    ).agg(
        assessment=('assessment', 'max') # Take the worst-case BI-RADS score
    ).reset_index()

    # 5. Pair CC and MLO views
    print("Pairing CC and MLO views for each patient...")
    paired = img_level.pivot_table(
        index=['patient_id', 'left or right breast'], 
        columns='image view', 
        values=['jpg_path', 'assessment'], 
        aggfunc='first'
    )
    
    # Flatten the multi-index columns created by pivot_table
    paired.columns = [f"{col[0]}_{col[1]}" for col in paired.columns]
    paired = paired.reset_index()

    # Drop any records that don't have BOTH a CC and an MLO view
    paired = paired.dropna(subset=['jpg_path_CC', 'jpg_path_MLO'])

    # 6. Format for the Spatial Alignment Pipeline
    final_df = paired[['patient_id', 'jpg_path_CC', 'jpg_path_MLO', 'assessment_CC', 'left or right breast']].copy()
    final_df = final_df.rename(columns={
        'patient_id': 'patient_id',
        'jpg_path_CC': 'cc_image_path',
        'jpg_path_MLO': 'mlo_image_path',
        'assessment_CC': 'birads_label',
        'left or right breast': 'breast_side'
    })

    final_df['birads_label'] = final_df['birads_label'].astype(int)
    final_df.to_csv(output_csv, index=False)
    print(f"\nSuccess! Cleaned and aggregated manifest saved to {output_csv}")
    print(f"Total valid CC/MLO pairs ready for STN: {len(final_df)}")

In [36]:
import os

print("base_dir exists:", os.path.exists("birads_dataset"))
print("jpeg_dir exists:", os.path.exists("birads_dataset/jpeg"))
print("mass csv exists:", os.path.exists("birads_dataset/mass_case_description_train_set.csv"))
print("calc csv exists:", os.path.exists("birads_dataset/calc_case_description_train_set.csv"))

base_dir exists: True
jpeg_dir exists: True
mass csv exists: False
calc csv exists: False


In [37]:
generate_paired_manifest()

Scanning for JPEG images...
Pairing CC and MLO views for each patient...

Success! Cleaned and aggregated manifest saved to birads_dataset/clean_pairs.csv
Total valid CC/MLO pairs ready for STN: 1062


In [38]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error, confusion_matrix
from sklearn.metrics import f1_score, cohen_kappa_score, roc_auc_score
import timm
from torch.utils.data import WeightedRandomSampler

In [39]:
df = pd.read_csv("birads_dataset/clean_pairs.csv")

print(df.head())
print("Rows:", len(df))
print("Labels:", sorted(df["birads_label"].unique()))

  patient_id                                      cc_image_path  \
0    P_00001  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.3...   
1    P_00004  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.8...   
2    P_00005  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.4...   
3    P_00007  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.2...   
4    P_00008  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.4...   

                                      mlo_image_path  birads_label breast_side  
0  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.3...             4        LEFT  
1  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.2...             4        LEFT  
2  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.2...             3       RIGHT  
3  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.1...             4        LEFT  
4  birads_dataset\jpeg\1.3.6.1.4.1.9590.100.1.2.1...             2        LEFT  
Rows: 1062
Labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [40]:
counts = df["birads_label"].value_counts()
valid_labels = counts[counts >= 2].index

df = df[df["birads_label"].isin(valid_labels)].copy()
df = df[df["birads_label"] != 0].copy()
df = df[df["birads_label"] != 1].copy()
print(df["birads_label"].value_counts().sort_index())

birads_label
2    138
3    140
4    507
5    184
Name: count, dtype: int64


In [41]:
# patient-level split
patient_labels = (
    df.groupby("patient_id")["birads_label"]
    .max()
    .reset_index()
)

train_patients, temp_patients = train_test_split(
    patient_labels,
    test_size=0.2,
    random_state=42,
    stratify=patient_labels["birads_label"]
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.5,
    random_state=42,
    stratify=temp_patients["birads_label"]
)

train_ids = set(train_patients["patient_id"])
val_ids = set(val_patients["patient_id"])
test_ids = set(test_patients["patient_id"])

train_df = df[df["patient_id"].isin(train_ids)].copy()
val_df   = df[df["patient_id"].isin(val_ids)].copy()
test_df  = df[df["patient_id"].isin(test_ids)].copy()

print("Train rows:", len(train_df), "| patients:", train_df["patient_id"].nunique())
print("Val rows:", len(val_df), "| patients:", val_df["patient_id"].nunique())
print("Test rows:", len(test_df), "| patients:", test_df["patient_id"].nunique())

print("Patient overlap checks:")
print("train ∩ val:", len(train_ids & val_ids))
print("train ∩ test:", len(train_ids & test_ids))
print("val ∩ test:", len(val_ids & test_ids))

Train rows: 776 | patients: 709
Val rows: 96 | patients: 89
Test rows: 97 | patients: 89
Patient overlap checks:
train ∩ val: 0
train ∩ test: 0
val ∩ test: 0


In [42]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.5.1+cu121
12.1
True
1


In [43]:
os.makedirs("birads_dataset/splits", exist_ok=True)

train_df.to_csv("birads_dataset/splits/train.csv", index=False)
val_df.to_csv("birads_dataset/splits/val.csv", index=False)
test_df.to_csv("birads_dataset/splits/test.csv", index=False)

In [44]:
classes = sorted(df["birads_label"].unique())
label_map = {c: i for i, c in enumerate(classes)}
inv_label_map = {i: c for c, i in label_map.items()}

print("label_map:", label_map)

for split_df in [train_df, val_df, test_df]:
    split_df["ordinal_label"] = split_df["birads_label"].map(label_map)
    # 0 = benign (BI-RADS 2–3), 1 = malignant (4–5)
    split_df["binary_label"] = (split_df["birads_label"] >= 4).astype(int)

label_map: {np.int64(2): 0, np.int64(3): 1, np.int64(4): 2, np.int64(5): 3}


In [45]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [46]:
class SingleViewDataset(Dataset):
    def __init__(self, dataframe, image_col, transform=None, label_col="ordinal_label"):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_col = image_col
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row[self.image_col]).convert("RGB")
        label = int(row[self.label_col])

        if self.transform:
            img = self.transform(img)

        return img, label

In [47]:
class PairedDataset(Dataset):
    def __init__(self, dataframe, transform=None, label_col="ordinal_label"):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        cc_path = str(row["cc_image_path"]).replace("\\", "/")
        mlo_path = str(row["mlo_image_path"]).replace("\\", "/")

        cc_img = Image.open(cc_path).convert("RGB")
        mlo_img = Image.open(mlo_path).convert("RGB")
        label = int(row[self.label_col])

        if self.transform:
            cc_img = self.transform(cc_img)
            mlo_img = self.transform(mlo_img)

        return cc_img, mlo_img, label

In [48]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [49]:
class_counts = train_df["ordinal_label"].value_counts().sort_index()
print(class_counts)

class_weights = 1.0 / class_counts
sample_weights = train_df["ordinal_label"].map(class_weights).values
sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

ordinal_label
0    112
1    112
2    406
3    146
Name: count, dtype: int64


In [50]:
batch_size = 16

cc_train_ds = SingleViewDataset(train_df, image_col="cc_image_path", transform=train_transform)
cc_val_ds   = SingleViewDataset(val_df, image_col="cc_image_path", transform=val_transform)
cc_test_ds  = SingleViewDataset(test_df, image_col="cc_image_path", transform=val_transform)

mlo_train_ds = SingleViewDataset(train_df, image_col="mlo_image_path", transform=train_transform)
mlo_val_ds   = SingleViewDataset(val_df, image_col="mlo_image_path", transform=val_transform)
mlo_test_ds  = SingleViewDataset(test_df, image_col="mlo_image_path", transform=val_transform)

paired_test_ds = PairedDataset(test_df, transform=val_transform)

cc_train_loader = DataLoader(cc_train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
cc_val_loader   = DataLoader(cc_val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
cc_test_loader  = DataLoader(cc_test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

mlo_train_loader = DataLoader(mlo_train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
mlo_val_loader   = DataLoader(mlo_val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
mlo_test_loader  = DataLoader(mlo_test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

paired_test_loader = DataLoader(paired_test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

# --- Binary (benign vs malignant): loaders with class-balanced sampling ---
binary_counts = train_df["binary_label"].value_counts().sort_index()
binary_class_weights = 1.0 / binary_counts
binary_sample_weights = torch.DoubleTensor(
    train_df["binary_label"].map(binary_class_weights).values
)
binary_sampler = WeightedRandomSampler(
    weights=binary_sample_weights,
    num_samples=len(binary_sample_weights),
    replacement=True,
)

cc_train_ds_bin = SingleViewDataset(
    train_df, image_col="cc_image_path", transform=train_transform, label_col="binary_label"
)
cc_val_ds_bin = SingleViewDataset(
    val_df, image_col="cc_image_path", transform=val_transform, label_col="binary_label"
)
cc_test_ds_bin = SingleViewDataset(
    test_df, image_col="cc_image_path", transform=val_transform, label_col="binary_label"
)
mlo_train_ds_bin = SingleViewDataset(
    train_df, image_col="mlo_image_path", transform=train_transform, label_col="binary_label"
)
mlo_val_ds_bin = SingleViewDataset(
    val_df, image_col="mlo_image_path", transform=val_transform, label_col="binary_label"
)
mlo_test_ds_bin = SingleViewDataset(
    test_df, image_col="mlo_image_path", transform=val_transform, label_col="binary_label"
)

cc_train_loader_bin = DataLoader(
    cc_train_ds_bin, batch_size=batch_size, sampler=binary_sampler, num_workers=0
)
cc_val_loader_bin = DataLoader(cc_val_ds_bin, batch_size=batch_size, shuffle=False, num_workers=0)
cc_test_loader_bin = DataLoader(cc_test_ds_bin, batch_size=batch_size, shuffle=False, num_workers=0)

mlo_train_loader_bin = DataLoader(
    mlo_train_ds_bin, batch_size=batch_size, sampler=binary_sampler, num_workers=0
)
mlo_val_loader_bin = DataLoader(mlo_val_ds_bin, batch_size=batch_size, shuffle=False, num_workers=0)
mlo_test_loader_bin = DataLoader(mlo_test_ds_bin, batch_size=batch_size, shuffle=False, num_workers=0)

paired_train_ds_bin = PairedDataset(train_df, transform=train_transform, label_col="binary_label")
paired_val_ds_bin = PairedDataset(val_df, transform=val_transform, label_col="binary_label")
paired_test_ds_bin = PairedDataset(test_df, transform=val_transform, label_col="binary_label")

paired_train_loader_bin = DataLoader(
    paired_train_ds_bin, batch_size=batch_size, sampler=binary_sampler, num_workers=0
)
paired_val_loader_bin = DataLoader(
    paired_val_ds_bin, batch_size=batch_size, shuffle=False, num_workers=0
)
paired_test_loader_bin = DataLoader(
    paired_test_ds_bin, batch_size=batch_size, shuffle=False, num_workers=0
)

print("len(cc_val_ds):", len(cc_val_ds))
print("len(cc_val_ds.df):", len(cc_val_ds.df))

len(cc_val_ds): 96
len(cc_val_ds.df): 96


In [51]:
num_classes = len(classes)

def ordinal_loss(predictions, targets):
    num_classes = predictions.size(1) + 1
    levels = torch.arange(num_classes - 1).to(predictions.device)
    binary_labels = (targets.view(-1, 1) > levels).float()
    return nn.BCEWithLogitsLoss()(predictions, binary_labels)

def coral_logits_to_label(logits):
    probas = torch.sigmoid(logits)
    return torch.sum(probas > 0.5, dim=1)

def coral_logits_to_proba(logits):
    cum_probs = torch.sigmoid(logits)
    batch_size = logits.size(0)
    k = logits.size(1) + 1

    probs = torch.zeros(batch_size, k, device=logits.device)
    probs[:, 0] = 1 - cum_probs[:, 0]

    for i in range(1, k - 1):
        probs[:, i] = cum_probs[:, i - 1] - cum_probs[:, i]

    probs[:, k - 1] = cum_probs[:, k - 2]
    return probs

In [52]:
vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")

class ViTOrdinal(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        self.head = nn.Linear(self.vit.config.hidden_size, num_classes - 1)

    def forward(self, x):
        outputs = self.vit(pixel_values=x)
        cls = outputs.last_hidden_state[:, 0]
        return self.head(cls)

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 4998.87it/s]


In [53]:
def train_one_epoch(model, loader, optimizer, device, num_classes):
    model.train()
    running_loss = 0.0
    total = 0
    correct = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = ordinal_loss(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        total += imgs.size(0)
        with torch.no_grad():
            preds = coral_logits_to_label(logits)
            correct += (preds == labels).sum().item()

    return running_loss / total, correct / total


In [54]:
@torch.no_grad()
def evaluate(model, loader, device, return_probs=False):
    model.eval()
    all_preds = []
    all_labels = []
    prob_chunks = [] if return_probs else None

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        logits = model(imgs)
        preds = coral_logits_to_label(logits)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        if return_probs:
            prob_chunks.append(coral_logits_to_proba(logits).cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    if return_probs:
        return acc, mae, all_labels, all_preds, np.vstack(prob_chunks)
    return acc, mae, all_labels, all_preds


In [55]:
def adjacent_accuracy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs(y_true - y_pred) <= 1)

In [56]:
from collections import Counter

def ordinal_indices_to_birads(y_ord, inv_map=None):
    inv = inv_label_map if inv_map is None else inv_map
    y = np.asarray(y_ord, dtype=int)
    return np.array([int(inv[int(i)]) for i in y])


def birads_to_benign_malignant(birads):
    b = np.asarray(birads, dtype=int)
    return (b >= 4).astype(int)


def malignant_prob_from_ordinal_probs(probs, inv_map=None):
    inv = inv_label_map if inv_map is None else inv_map
    k = probs.shape[1]
    w = np.array([1.0 if int(inv[j]) >= 4 else 0.0 for j in range(k)])
    return probs @ w


def report_ordinal_and_binary(title, y_true_ord, y_pred_ord, y_score_malignant=None, inv_map=None):
    y_true_ord = np.asarray(y_true_ord, dtype=int)
    y_pred_ord = np.asarray(y_pred_ord, dtype=int)
    true_b = ordinal_indices_to_birads(y_true_ord, inv_map)
    pred_b = ordinal_indices_to_birads(y_pred_ord, inv_map)

    print(f"=== {title} — ordinal (BI-RADS class index) ===")
    print("True counts:", Counter(y_true_ord))
    print("Pred counts:", Counter(y_pred_ord))
    print(
        f"Accuracy: {accuracy_score(y_true_ord, y_pred_ord):.4f} | "
        f"MAE: {mean_absolute_error(y_true_ord, y_pred_ord):.4f}"
    )
    print("F1 macro:", f1_score(y_true_ord, y_pred_ord, average="macro"))
    print("F1 weighted:", f1_score(y_true_ord, y_pred_ord, average="weighted"))
    print("Adjacent accuracy:", adjacent_accuracy(y_true_ord, y_pred_ord))
    print("QWK:", cohen_kappa_score(y_true_ord, y_pred_ord, weights="quadratic"))

    y_true_bm = birads_to_benign_malignant(true_b)
    y_pred_bm = birads_to_benign_malignant(pred_b)
    print(f"=== {title} — benign (BI-RADS 2–3) vs malignant (4–5) ===")
    print("True counts:", Counter(y_true_bm))
    print("Pred counts:", Counter(y_pred_bm))
    print("Binary accuracy:", accuracy_score(y_true_bm, y_pred_bm))
    print(
        "Binary F1 (malignant positive):",
        f1_score(y_true_bm, y_pred_bm, pos_label=1, zero_division=0),
    )
    print("Binary confusion matrix [rows=true 0=benign,1=malignant]:")
    print(confusion_matrix(y_true_bm, y_pred_bm))

    if y_score_malignant is not None:
        y_score_malignant = np.asarray(y_score_malignant, dtype=float)
        if len(np.unique(y_true_bm)) > 1:
            try:
                print(
                    "Binary ROC-AUC (prob sum for 4+5):",
                    roc_auc_score(y_true_bm, y_score_malignant),
                )
            except ValueError as e:
                print("Binary ROC-AUC: n/a", e)
    print()


### Part 1 — Ordinal classifier (CORAL)

We **continue** with the **ordinal** setup here: train and validate **CC** and **MLO** ViTs with the same CORAL head and ordinal loss. Checkpoints are saved from validation **MAE** (lower is better). After both views are trained, we evaluate on the validation set, then on the test set, and we build the **CC+MLO probability ensemble** (average of the two views’ ordinal distributions).

Reporting includes **ordinal** metrics (accuracy, MAE, macro/weighted F1, adjacent accuracy, quadratic Cohen’s kappa) and **derived** benign-vs-malignant stats from the **predicted** BI-RADS bins (not a separately trained binary head).

In [57]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cc_model = ViTOrdinal(num_classes=num_classes).to(device)
cc_optimizer = torch.optim.AdamW([
    {"params": cc_model.vit.parameters(), "lr": 1e-5},
    {"params": cc_model.head.parameters(), "lr": 5e-4},
], weight_decay=1e-4)

best_val_mae = float("inf")
best_cc_path = "birads_dataset/cc_vit_ordinal.pt"

epochs = 20
patience = 5
epochs_without_improvement = 0

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(cc_model, cc_train_loader, cc_optimizer, device, num_classes)
    val_acc, val_mae, _, _ = evaluate(cc_model, cc_val_loader, device)

    print(f"[CC] Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | val_mae={val_mae:.4f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        epochs_without_improvement = 0
        torch.save(cc_model.state_dict(), best_cc_path)
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

print("Best CC model saved to:", best_cc_path)

cuda


Loading weights: 100%|██████████| 200/200 [00:00<00:00, 9521.91it/s]


[CC] Epoch 1/20 | train_loss=0.6119 | train_acc=0.2461 | val_acc=0.4062 | val_mae=0.7604
[CC] Epoch 2/20 | train_loss=0.5334 | train_acc=0.3196 | val_acc=0.4375 | val_mae=0.6771
[CC] Epoch 3/20 | train_loss=0.4374 | train_acc=0.4729 | val_acc=0.3438 | val_mae=0.8958
[CC] Epoch 4/20 | train_loss=0.4300 | train_acc=0.4948 | val_acc=0.3750 | val_mae=0.8333
[CC] Epoch 5/20 | train_loss=0.3775 | train_acc=0.5735 | val_acc=0.4062 | val_mae=0.8021
[CC] Epoch 6/20 | train_loss=0.3546 | train_acc=0.6134 | val_acc=0.4375 | val_mae=0.7083
[CC] Epoch 7/20 | train_loss=0.2852 | train_acc=0.6881 | val_acc=0.3750 | val_mae=0.7917
Early stopping triggered at epoch 7
Best CC model saved to: birads_dataset/cc_vit_ordinal.pt


In [58]:
from collections import Counter

cc_model.load_state_dict(torch.load(best_cc_path, map_location=device))

val_acc, val_mae, y_true, y_pred = evaluate(cc_model, cc_val_loader, device)
_ = val_acc, val_mae
report_ordinal_and_binary("CC view — validation", y_true, y_pred)


C:\Users\Roofu\AppData\Local\Temp\ipykernel_7576\4255971061.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cc_model.load_state_dict(torch.load(best_cc_path, map_locatio

=== CC view — validation — ordinal (BI-RADS class index) ===
True counts: Counter({np.int64(2): 49, np.int64(3): 19, np.int64(0): 14, np.int64(1): 14})
Pred counts: Counter({np.int64(2): 65, np.int64(1): 26, np.int64(0): 5})
Accuracy: 0.4375 | MAE: 0.6771
F1 macro: 0.2824561403508772
F1 weighted: 0.3886330409356726
Adjacent accuracy: 0.8854166666666666
QWK: 0.2611464968152867
=== CC view — validation — benign (BI-RADS 2–3) vs malignant (4–5) ===
True counts: Counter({np.int64(1): 68, np.int64(0): 28})
Pred counts: Counter({np.int64(1): 65, np.int64(0): 31})
Binary accuracy: 0.65625
Binary F1 (malignant positive): 0.7518796992481203
Binary confusion matrix [rows=true 0=benign,1=malignant]:
[[13 15]
 [18 50]]



In [59]:
# Ordinal (4-class) confusion matrix — CC validation
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm)
cm_df


,0,1,2,3
0,3,4,7,0
1,2,4,8,0
2,0,14,35,0
3,0,4,15,0


In [60]:
mlo_model = ViTOrdinal(num_classes=num_classes).to(device)
mlo_optimizer = torch.optim.AdamW([
    {"params": mlo_model.vit.parameters(), "lr": 1e-5},
    {"params": mlo_model.head.parameters(), "lr": 5e-4},
], weight_decay=1e-4)

best_val_mae = float("inf")
best_mlo_path = "birads_dataset/mlo_vit_ordinal.pt"

epochs = 20
patience = 5
epochs_without_improvement = 0

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(mlo_model, mlo_train_loader, mlo_optimizer, device, num_classes)
    val_acc, val_mae, _, _ = evaluate(mlo_model, mlo_val_loader, device)

    print(f"[MLO] Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | val_mae={val_mae:.4f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        epochs_without_improvement = 0
        torch.save(mlo_model.state_dict(), best_mlo_path)
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

print("Best MLO model saved to:", best_mlo_path)

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 8693.49it/s]


[MLO] Epoch 1/20 | train_loss=0.5887 | train_acc=0.2925 | val_acc=0.3542 | val_mae=0.7396
[MLO] Epoch 2/20 | train_loss=0.5110 | train_acc=0.3299 | val_acc=0.3958 | val_mae=0.6562
[MLO] Epoch 3/20 | train_loss=0.4646 | train_acc=0.4201 | val_acc=0.4896 | val_mae=0.5938
[MLO] Epoch 4/20 | train_loss=0.4216 | train_acc=0.4794 | val_acc=0.4688 | val_mae=0.6354
[MLO] Epoch 5/20 | train_loss=0.3769 | train_acc=0.5399 | val_acc=0.4896 | val_mae=0.6042
[MLO] Epoch 6/20 | train_loss=0.3464 | train_acc=0.6044 | val_acc=0.4271 | val_mae=0.7083
[MLO] Epoch 7/20 | train_loss=0.2994 | train_acc=0.6714 | val_acc=0.5000 | val_mae=0.6042
[MLO] Epoch 8/20 | train_loss=0.3078 | train_acc=0.6534 | val_acc=0.5417 | val_mae=0.5312
[MLO] Epoch 9/20 | train_loss=0.2714 | train_acc=0.7152 | val_acc=0.4688 | val_mae=0.6354
[MLO] Epoch 10/20 | train_loss=0.2167 | train_acc=0.7822 | val_acc=0.3854 | val_mae=0.7292
[MLO] Epoch 11/20 | train_loss=0.2108 | train_acc=0.7719 | val_acc=0.5000 | val_mae=0.5938
[MLO] Ep

In [61]:
mlo_model.load_state_dict(torch.load(best_mlo_path, map_location=device))

val_acc, val_mae, y_true, y_pred = evaluate(mlo_model, mlo_val_loader, device)
_ = val_acc, val_mae
report_ordinal_and_binary("MLO view — validation", y_true, y_pred)


C:\Users\Roofu\AppData\Local\Temp\ipykernel_7576\4213528074.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mlo_model.load_state_dict(torch.load(best_mlo_path, map_locat

=== MLO view — validation — ordinal (BI-RADS class index) ===
True counts: Counter({np.int64(2): 49, np.int64(3): 19, np.int64(0): 14, np.int64(1): 14})
Pred counts: Counter({np.int64(2): 42, np.int64(3): 26, np.int64(1): 18, np.int64(0): 10})
Accuracy: 0.5417 | MAE: 0.5312
F1 macro: 0.5208600427350427
F1 weighted: 0.5536569622507123
Adjacent accuracy: 0.9375
QWK: 0.5994020926756353
=== MLO view — validation — benign (BI-RADS 2–3) vs malignant (4–5) ===
True counts: Counter({np.int64(1): 68, np.int64(0): 28})
Pred counts: Counter({np.int64(1): 68, np.int64(0): 28})
Binary accuracy: 0.7916666666666666
Binary F1 (malignant positive): 0.8529411764705882
Binary confusion matrix [rows=true 0=benign,1=malignant]:
[[18 10]
 [10 58]]



In [62]:
cc_model.load_state_dict(torch.load(best_cc_path, map_location=device))
mlo_model.load_state_dict(torch.load(best_mlo_path, map_location=device))

cc_test_acc, cc_test_mae, cc_y_true, cc_y_pred, cc_test_probs = evaluate(
    cc_model, cc_test_loader, device, return_probs=True
)
mlo_test_acc, mlo_test_mae, mlo_y_true, mlo_y_pred, mlo_test_probs = evaluate(
    mlo_model, mlo_test_loader, device, return_probs=True
)

cc_mal = malignant_prob_from_ordinal_probs(cc_test_probs)
mlo_mal = malignant_prob_from_ordinal_probs(mlo_test_probs)

report_ordinal_and_binary(
    "CC view — test", cc_y_true, cc_y_pred, y_score_malignant=cc_mal
)
report_ordinal_and_binary(
    "MLO view — test", mlo_y_true, mlo_y_pred, y_score_malignant=mlo_mal
)


C:\Users\Roofu\AppData\Local\Temp\ipykernel_7576\1705100153.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cc_model.load_state_dict(torch.load(best_cc_path, map_locatio

=== CC view — test — ordinal (BI-RADS class index) ===
True counts: Counter({np.int64(2): 52, np.int64(3): 19, np.int64(1): 14, np.int64(0): 12})
Pred counts: Counter({np.int64(2): 61, np.int64(1): 32, np.int64(0): 4})
Accuracy: 0.5052 | MAE: 0.5670
F1 macro: 0.35633897652943436
F1 weighted: 0.47275915002994806
Adjacent accuracy: 0.9278350515463918
QWK: 0.3911580096424998
=== CC view — test — benign (BI-RADS 2–3) vs malignant (4–5) ===
True counts: Counter({np.int64(1): 71, np.int64(0): 26})
Pred counts: Counter({np.int64(1): 61, np.int64(0): 36})
Binary accuracy: 0.711340206185567
Binary F1 (malignant positive): 0.7878787878787878
Binary confusion matrix [rows=true 0=benign,1=malignant]:
[[17  9]
 [19 52]]
Binary ROC-AUC (prob sum for 4+5): 0.7183098591549295

=== MLO view — test — ordinal (BI-RADS class index) ===
True counts: Counter({np.int64(2): 52, np.int64(3): 19, np.int64(1): 14, np.int64(0): 12})
Pred counts: Counter({np.int64(2): 37, np.int64(3): 31, np.int64(1): 25, np.int64

In [63]:
@torch.no_grad()
def evaluate_ensemble(cc_model, mlo_model, loader, device):
    cc_model.eval()
    mlo_model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    for cc_imgs, mlo_imgs, labels in loader:
        cc_imgs = cc_imgs.to(device)
        mlo_imgs = mlo_imgs.to(device)

        cc_logits = cc_model(cc_imgs)
        mlo_logits = mlo_model(mlo_imgs)

        cc_probs = coral_logits_to_proba(cc_logits)
        mlo_probs = coral_logits_to_proba(mlo_logits)

        avg_probs = (cc_probs + mlo_probs) / 2.0
        preds = torch.argmax(avg_probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.append(avg_probs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    all_probs = np.vstack(all_probs)

    return acc, mae, all_labels, all_preds, all_probs


In [64]:
ens_acc, ens_mae, ens_y_true, ens_y_pred, ens_probs = evaluate_ensemble(
    cc_model, mlo_model, paired_test_loader, device
)

ens_mal = malignant_prob_from_ordinal_probs(ens_probs)
print(f"Ensemble test — headline: acc={ens_acc:.4f} MAE={ens_mae:.4f}\n")
report_ordinal_and_binary(
    "Ensemble (CC+MLO avg probs) — test",
    ens_y_true,
    ens_y_pred,
    y_score_malignant=ens_mal,
)


Ensemble test — headline: acc=0.4227 MAE=0.7320

=== Ensemble (CC+MLO avg probs) — test — ordinal (BI-RADS class index) ===
True counts: Counter({np.int64(2): 52, np.int64(3): 19, np.int64(1): 14, np.int64(0): 12})
Pred counts: Counter({np.int64(2): 39, np.int64(3): 33, np.int64(1): 15, np.int64(0): 10})
Accuracy: 0.4227 | MAE: 0.7320
F1 macro: 0.36851510558407113
F1 weighted: 0.43182572848410034
Adjacent accuracy: 0.865979381443299
QWK: 0.37580437580437576
=== Ensemble (CC+MLO avg probs) — test — benign (BI-RADS 2–3) vs malignant (4–5) ===
True counts: Counter({np.int64(1): 71, np.int64(0): 26})
Pred counts: Counter({np.int64(1): 72, np.int64(0): 25})
Binary accuracy: 0.7010309278350515
Binary F1 (malignant positive): 0.7972027972027972
Binary confusion matrix [rows=true 0=benign,1=malignant]:
[[11 15]
 [14 57]]
Binary ROC-AUC (prob sum for 4+5): 0.7416034669555797



In [65]:
print("First 20 true:", ens_y_true[:20])
print("First 20 pred:", ens_y_pred[:20])

errors = np.abs(np.array(ens_y_true) - np.array(ens_y_pred))
print("Error counts:", np.unique(errors, return_counts=True))
print("Accuracy:", np.mean(np.array(ens_y_true) == np.array(ens_y_pred)))
print("MAE:", errors.mean())

First 20 true: [np.int64(1), np.int64(0), np.int64(0), np.int64(3), np.int64(1), np.int64(1), np.int64(3), np.int64(2), np.int64(3), np.int64(2), np.int64(1), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(1), np.int64(1), np.int64(2), np.int64(2)]
First 20 pred: [np.int64(2), np.int64(1), np.int64(3), np.int64(3), np.int64(2), np.int64(3), np.int64(3), np.int64(2), np.int64(1), np.int64(3), np.int64(2), np.int64(3), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(3), np.int64(2), np.int64(2), np.int64(0)]
Error counts: (array([0, 1, 2, 3]), array([41, 43, 11,  2]))
Accuracy: 0.422680412371134
MAE: 0.7319587628865979


In [66]:
ens_f1_macro = f1_score(ens_y_true, ens_y_pred, average="macro")
ens_f1_weighted = f1_score(ens_y_true, ens_y_pred, average="weighted")
ens_adj_acc = adjacent_accuracy(ens_y_true, ens_y_pred)
ens_qwk = cohen_kappa_score(ens_y_true, ens_y_pred, weights="quadratic")

print("Ensemble — ordinal F1 (macro):", ens_f1_macro)
print("Ensemble — ordinal F1 (weighted):", ens_f1_weighted)
print("Ensemble — ordinal adjacent accuracy:", ens_adj_acc)
print("Ensemble — ordinal QWK:", ens_qwk)


Ensemble — ordinal F1 (macro): 0.36851510558407113
Ensemble — ordinal F1 (weighted): 0.43182572848410034
Ensemble — ordinal adjacent accuracy: 0.865979381443299
Ensemble — ordinal QWK: 0.37580437580437576


### Part 2 — Binary classifiers (different training target)

Here we **diverge** from the ordinal objective: labels are **binary** (0 = benign BI-RADS 2–3, 1 = malignant 4–5). We train **three** models with **BCE** on logits:

- **CC-only** and **MLO-only** ViTs (single view, one logit each).
- **Fused CC+MLO**: two ViT encoders (one per view), **concatenated** CLS embeddings, then a linear layer to one logit — trained end-to-end on the binary label.

Early stopping uses validation **ROC-AUC**. This section is **additive**: the ordinal models and checkpoints above are unchanged; these are **separate** weights (`*_vit_binary.pt`). Use this block when you care about **direct** malignant-vs-benign performance rather than full 4-way BI-RADS accuracy.

In [67]:
# --- Binary classification (benign BI-RADS 2–3 vs malignant 4–5): CC, MLO, and fused CC+MLO ---
# Run after ordinal training so `device` is defined. Re-run the label + DataLoader cells if splits changed.

import torch.nn.functional as F


class ViTBinary(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        self.head = nn.Linear(self.vit.config.hidden_size, 1)

    def forward(self, x):
        cls = self.vit(pixel_values=x).last_hidden_state[:, 0]
        return self.head(cls)


class ViTPairedBinary(nn.Module):
    """CC + MLO: two ViT encoders, concatenate CLS, single logit."""

    def __init__(self):
        super().__init__()
        self.cc_vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        self.mlo_vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        h = self.cc_vit.config.hidden_size
        self.head = nn.Linear(h * 2, 1)

    def forward(self, cc, mlo):
        cc_e = self.cc_vit(pixel_values=cc).last_hidden_state[:, 0]
        mlo_e = self.mlo_vit(pixel_values=mlo).last_hidden_state[:, 0]
        return self.head(torch.cat([cc_e, mlo_e], dim=1))


def train_one_epoch_binary(model, loader, optimizer, device, paired=False):
    model.train()
    running_loss = 0.0
    n = 0
    for batch in loader:
        optimizer.zero_grad()
        if paired:
            cc, mlo, labels = batch
            cc, mlo = cc.to(device), mlo.to(device)
            labels = labels.to(device).float()
            logits = model(cc, mlo).squeeze(-1)
        else:
            imgs, labels = batch
            imgs = imgs.to(device)
            labels = labels.to(device).float()
            logits = model(imgs).squeeze(-1)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * labels.size(0)
        n += labels.size(0)
    return running_loss / max(n, 1)


@torch.no_grad()
def evaluate_binary(model, loader, device, paired=False):
    model.eval()
    all_labels, all_scores = [], []
    for batch in loader:
        if paired:
            cc, mlo, labels = batch
            cc, mlo = cc.to(device), mlo.to(device)
            logits = model(cc, mlo).squeeze(-1)
        else:
            imgs, labels = batch
            imgs = imgs.to(device)
            logits = model(imgs).squeeze(-1)
        if torch.is_tensor(labels):
            labels = labels.cpu().numpy()
        all_labels.extend(np.asarray(labels).astype(float))
        all_scores.extend(torch.sigmoid(logits).cpu().numpy())
    all_labels = np.asarray(all_labels, dtype=float)
    all_scores = np.asarray(all_scores, dtype=float)
    all_preds = (all_scores >= 0.5).astype(int)
    acc = accuracy_score(all_labels, all_preds)
    return acc, all_labels, all_preds, all_scores


def train_binary_model(name, model, train_loader, val_loader, save_path, device, paired=False, epochs=20, patience=5):
    if paired:
        opt_params = [
            {"params": list(model.cc_vit.parameters()) + list(model.mlo_vit.parameters()), "lr": 1e-5},
            {"params": model.head.parameters(), "lr": 5e-4},
        ]
    else:
        opt_params = [
            {"params": model.vit.parameters(), "lr": 1e-5},
            {"params": model.head.parameters(), "lr": 5e-4},
        ]
    optimizer = torch.optim.AdamW(opt_params, weight_decay=1e-4)
    best_auc = -1.0
    stall = 0
    for epoch in range(epochs):
        tr_loss = train_one_epoch_binary(model, train_loader, optimizer, device, paired=paired)
        _, yv, _, sv = evaluate_binary(model, val_loader, device, paired=paired)
        yv_int = yv.astype(int)
        if len(np.unique(yv_int)) > 1:
            val_auc = roc_auc_score(yv_int, sv)
        else:
            val_auc = 0.0
        print(
            f"[{name}] Epoch {epoch + 1}/{epochs} | train_loss={tr_loss:.4f} | val_auc={val_auc:.4f}"
        )
        if val_auc > best_auc:
            best_auc = val_auc
            stall = 0
            torch.save(model.state_dict(), save_path)
        else:
            stall += 1
        if stall >= patience:
            print(f"[{name}] Early stopping at epoch {epoch + 1}")
            break
    print(f"[{name}] Best checkpoint: {save_path} (best val_auc={best_auc:.4f})")
    return save_path


def report_binary_eval(title, y_true, y_pred, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    print(f"=== {title} ===")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("F1 (malignant):", f1_score(y_true, y_pred, pos_label=1, zero_division=0))
    if len(np.unique(y_true)) > 1:
        print("ROC-AUC:", roc_auc_score(y_true, y_score))
    print("Confusion [true 0=benign, 1=malignant]:\n", confusion_matrix(y_true, y_pred))
    print()

In [68]:
# Train binary heads (CC, MLO, fused CC+MLO)
best_cc_bin_path = "birads_dataset/cc_vit_binary.pt"
best_mlo_bin_path = "birads_dataset/mlo_vit_binary.pt"
best_paired_bin_path = "birads_dataset/paired_vit_binary.pt"

cc_bin_model = ViTBinary().to(device)
train_binary_model(
    "CC-binary",
    cc_bin_model,
    cc_train_loader_bin,
    cc_val_loader_bin,
    best_cc_bin_path,
    device,
    paired=False,
)

mlo_bin_model = ViTBinary().to(device)
train_binary_model(
    "MLO-binary",
    mlo_bin_model,
    mlo_train_loader_bin,
    mlo_val_loader_bin,
    best_mlo_bin_path,
    device,
    paired=False,
)

paired_bin_model = ViTPairedBinary().to(device)
train_binary_model(
    "CC+MLO-binary",
    paired_bin_model,
    paired_train_loader_bin,
    paired_val_loader_bin,
    best_paired_bin_path,
    device,
    paired=True,
)

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 9521.91it/s]


[CC-binary] Epoch 1/20 | train_loss=0.6743 | val_auc=0.6539
[CC-binary] Epoch 2/20 | train_loss=0.6328 | val_auc=0.6822
[CC-binary] Epoch 3/20 | train_loss=0.6042 | val_auc=0.6896
[CC-binary] Epoch 4/20 | train_loss=0.5409 | val_auc=0.6696
[CC-binary] Epoch 5/20 | train_loss=0.4933 | val_auc=0.6670
[CC-binary] Epoch 6/20 | train_loss=0.4391 | val_auc=0.6665
[CC-binary] Epoch 7/20 | train_loss=0.4237 | val_auc=0.7043
[CC-binary] Epoch 8/20 | train_loss=0.3540 | val_auc=0.6345
[CC-binary] Epoch 9/20 | train_loss=0.2629 | val_auc=0.6838
[CC-binary] Epoch 10/20 | train_loss=0.2283 | val_auc=0.6980
[CC-binary] Epoch 11/20 | train_loss=0.2222 | val_auc=0.6759
[CC-binary] Epoch 12/20 | train_loss=0.1807 | val_auc=0.6938
[CC-binary] Early stopping at epoch 12
[CC-binary] Best checkpoint: birads_dataset/cc_vit_binary.pt (best val_auc=0.7043)


Loading weights: 100%|██████████| 200/200 [00:00<00:00, 9088.02it/s]


[MLO-binary] Epoch 1/20 | train_loss=0.6748 | val_auc=0.6733
[MLO-binary] Epoch 2/20 | train_loss=0.6166 | val_auc=0.7416
[MLO-binary] Epoch 3/20 | train_loss=0.5326 | val_auc=0.7300
[MLO-binary] Epoch 4/20 | train_loss=0.4582 | val_auc=0.7532
[MLO-binary] Epoch 5/20 | train_loss=0.4632 | val_auc=0.7437
[MLO-binary] Epoch 6/20 | train_loss=0.3631 | val_auc=0.7316
[MLO-binary] Epoch 7/20 | train_loss=0.3443 | val_auc=0.7232
[MLO-binary] Epoch 8/20 | train_loss=0.2720 | val_auc=0.7195
[MLO-binary] Epoch 9/20 | train_loss=0.3030 | val_auc=0.7043
[MLO-binary] Early stopping at epoch 9
[MLO-binary] Best checkpoint: birads_dataset/mlo_vit_binary.pt (best val_auc=0.7532)


Loading weights: 100%|██████████| 200/200 [00:00<00:00, 9089.01it/s]


[CC+MLO-binary] Epoch 1/20 | train_loss=0.6597 | val_auc=0.5809
[CC+MLO-binary] Epoch 2/20 | train_loss=0.5736 | val_auc=0.7237
[CC+MLO-binary] Epoch 3/20 | train_loss=0.5338 | val_auc=0.7027
[CC+MLO-binary] Epoch 4/20 | train_loss=0.4753 | val_auc=0.6991
[CC+MLO-binary] Epoch 5/20 | train_loss=0.4470 | val_auc=0.7363
[CC+MLO-binary] Epoch 6/20 | train_loss=0.4083 | val_auc=0.7043
[CC+MLO-binary] Epoch 7/20 | train_loss=0.3424 | val_auc=0.7012
[CC+MLO-binary] Epoch 8/20 | train_loss=0.2453 | val_auc=0.7164
[CC+MLO-binary] Epoch 9/20 | train_loss=0.1834 | val_auc=0.6560
[CC+MLO-binary] Epoch 10/20 | train_loss=0.1397 | val_auc=0.7195
[CC+MLO-binary] Early stopping at epoch 10
[CC+MLO-binary] Best checkpoint: birads_dataset/paired_vit_binary.pt (best val_auc=0.7363)


'birads_dataset/paired_vit_binary.pt'

In [69]:
# Test set — binary models (direct supervision on malignant vs benign)
cc_bin_model.load_state_dict(torch.load(best_cc_bin_path, map_location=device))
mlo_bin_model.load_state_dict(torch.load(best_mlo_bin_path, map_location=device))
paired_bin_model.load_state_dict(torch.load(best_paired_bin_path, map_location=device))

_, yt, yp, ys = evaluate_binary(cc_bin_model, cc_test_loader_bin, device, paired=False)
report_binary_eval("CC view — binary model — test", yt, yp, ys)

_, yt, yp, ys = evaluate_binary(mlo_bin_model, mlo_test_loader_bin, device, paired=False)
report_binary_eval("MLO view — binary model — test", yt, yp, ys)

_, yt, yp, ys = evaluate_binary(paired_bin_model, paired_test_loader_bin, device, paired=True)
report_binary_eval("Combined CC+MLO — binary model — test", yt, yp, ys)

C:\Users\Roofu\AppData\Local\Temp\ipykernel_7576\930108521.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cc_bin_model.load_state_dict(torch.load(best_cc_bin_path, map_

=== CC view — binary model — test ===
Accuracy: 0.4948453608247423
F1 (malignant): 0.5242718446601942
ROC-AUC: 0.724810400866739
Confusion [true 0=benign, 1=malignant]:
 [[21  5]
 [44 27]]

=== MLO view — binary model — test ===
Accuracy: 0.7319587628865979
F1 (malignant): 0.8142857142857143
ROC-AUC: 0.7518959913326111
Confusion [true 0=benign, 1=malignant]:
 [[14 12]
 [14 57]]

=== Combined CC+MLO — binary model — test ===
Accuracy: 0.7835051546391752
F1 (malignant): 0.8627450980392157
ROC-AUC: 0.7659804983748646
Confusion [true 0=benign, 1=malignant]:
 [[10 16]
 [ 5 66]]

